In [ ]:
import copy
from heapq import heappush, heappop

# Kích thước của puzzle (n=4 cho 15-puzzle)
n = 4

# 4 vị trí di chuyển tương ứng bottom, left, top, right
rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]

# Tạo một lớp hàng đợi ưu tiên
class priorityQueue:
    def __init__(self):
        self.heap = []

    # Thêm một phần tử mới vào hàng đợi
    def push(self, key):
        heappush(self.heap, key)

    # Lấy phần tử có ưu tiên cao nhất (chi phí thấp nhất)
    def pop(self):
        return heappop(self.heap)

    # Kiểm tra xem hàng đợi có trống không
    def empty(self):
        return not self.heap

# Cấu trúc của một nút trong cây tìm kiếm
class nodes:
    def __init__(self, parent, mats, empty_tile_posi, costs, levels):
        self.parent = parent # Nút cha
        self.mats = mats # Ma trận trạng thái hiện tại của puzzle
        self.empty_tile_posi = empty_tile_posi # Vị trí của ô trống (0)
        self.costs = costs # Chi phí heuristic (h(n))
        self.levels = levels # Chi phí đường đi (g(n))

    # Phương thức so sánh cho hàng đợi ưu tiên (ưu tiên nút có tổng chi phí thấp hơn)
    def __lt__(self, nxt):
        # Tổng chi phí f(n) = g(n) + h(n)
        # Ở đây, cost là h(n) và levels là g(n)
        return (self.costs + self.levels) < (nxt.costs + nxt.levels)

# Hàm tính chi phí heuristic (khoảng cách Manhattan)
def calculateCosts(mats, final) -> int:
    cost = 0
    # Tạo bảng tra cứu vị trí cuối cùng của từng ô
    final_pos_lookup = {}
    for r in range(n):
        for c in range(n):
            val = final[r][c]
            if val != 0: # Không tính ô trống
                final_pos_lookup[val] = (r, c)

    for r in range(n):
        for c in range(n):
            val = mats[r][c]
            if val != 0: # Không tính ô trống
                target_r, target_c = final_pos_lookup[val]
                # Cộng khoảng cách Manhattan
                cost += abs(r - target_r) + abs(c - target_c)
    return cost

# Tạo một nút mới
def newNodes(mats, empty_tile_posi, new_empty_tile_posi,
             levels, parent, final) -> nodes:

    # Sao chép dữ liệu từ ma trận cha sang ma trận hiện tại
    new_mats = copy.deepcopy(mats)

    # Di chuyển ô trống
    x1, y1 = empty_tile_posi
    x2, y2 = new_empty_tile_posi
    new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]

    # Tính chi phí heuristic mới
    costs = calculateCosts(new_mats, final)

    new_nodes = nodes(parent, new_mats, new_empty_tile_posi,
                      costs, levels)
    return new_nodes

# Hàm in ma trận N x N
def printMatsrix(mats):
    for i in range(n):
        for j in range(n):
            print("%2d " % (mats[i][j]), end=" ") # Định dạng để in số 2 chữ số
        print()

# Hàm kiểm tra xem (x, y) có phải là tọa độ ma trận hợp lệ không
def isSafe(x, y):
    return 0 <= x < n and 0 <= y < n


# In đường đi từ nút gốc đến nút đích
def printPath(root):
    if root is None:
        return
    printPath(root.parent)
    printMatsrix(root.mats)
    print()

# Hàm giải quyết bài toán N-puzzle bằng thuật toán A*
def solve(initial, empty_tile_posi, final):
    pq = priorityQueue()

    # Tạo nút gốc
    costs = calculateCosts(initial, final)
    root = nodes(None, initial, empty_tile_posi, costs, 0)

    # Thêm nút gốc vào hàng đợi ưu tiên
    pq.push(root)

    while not pq.empty():
        minimum = pq.pop()

        # Nếu đạt đến trạng thái mục tiêu
        if minimum.costs == 0:
            printPath(minimum)
            return

        # Tạo tất cả các nút con có thể có
        for i in range(4): # 4 hướng di chuyển
            new_tile_posi = [
                minimum.empty_tile_posi[0] + rows[i],
                minimum.empty_tile_posi[1] + cols[i],
            ]

            # Nếu vị trí mới hợp lệ
            if isSafe(new_tile_posi[0], new_tile_posi[1]):
                # Tạo nút con
                child = newNodes(minimum.mats,
                                 minimum.empty_tile_posi,
                                 new_tile_posi,
                                 minimum.levels + 1, # Tăng chi phí đường đi
                                 minimum,
                                 final)
                # Thêm nút con vào hàng đợi ưu tiên
                pq.push(child)
# Ví dụ cho 15-puzzle (n=4)
# Trạng thái ban đầu (ví dụ một trạng thái có thể giải được)
initial = [[1, 2, 3, 4],
           [5, 6, 7, 8],
           [9, 10, 11, 0],
           [13, 14, 15, 12]]
# Trạng thái mục tiêu
final = [[1, 2, 3, 4],
         [5, 6, 7, 8],
         [9, 10, 11, 12],
         [13, 14, 15, 0]]

# Vị trí ban đầu của ô trống (0)
empty_tile_posi = [2, 3]

solve(initial, empty_tile_posi, final)


 1   2   3   4  
 5   6   7   8  
 9  10  11   0  
13  14  15  12  

 1   2   3   4  
 5   6   7   8  
 9  10  11  12  
13  14  15   0  



In [ ]:
#A*
import heapq
import math
from itertools import combinations

class TSP_AStar:
    def __init__(self, distance_matrix, start=0):
        """
        distance_matrix: ma trận khoảng cách giữa các thành phố
        start: thành phố bắt đầu (mặc định 0)
        """
        self.n = len(distance_matrix)
        self.dist = distance_matrix
        self.start = start

    def heuristic(self, visited, current):
        """
        Ước lượng chi phí từ trạng thái hiện tại đến khi kết thúc:
        - Nếu đã thăm hết thành phố: khoảng cách từ current về start
        - Ngược lại: chi phí cây khung nhỏ nhất của các thành phố chưa thăm + khoảng cách gần nhất từ current đến các thành phố chưa thăm
        """
        unvisited = [city for city in range(self.n) if city not in visited]

        if not unvisited:  # Đã thăm hết
            return self.dist[current][self.start]

        # Tính MST trên tập unvisited + current (dùng Prim)
        nodes = unvisited + [current]
        mst_cost = self.mst_prim(nodes)

        # Heuristic đơn giản: chi phí MST + khoảng cách gần nhất từ current đến unvisited
        min_to_unvisited = min(self.dist[current][city] for city in unvisited)

        return mst_cost + min_to_unvisited

    def mst_prim(self, nodes):
        """Tính chi phí cây khung nhỏ nhất trên tập nodes"""
        if len(nodes) <= 1:
            return 0

        n_nodes = len(nodes)
        node_to_idx = {node: i for i, node in enumerate(nodes)}

        # Ma trận khoảng cách con
        dist_sub = [[0]*n_nodes for _ in range(n_nodes)]
        for i in range(n_nodes):
            for j in range(n_nodes):
                dist_sub[i][j] = self.dist[nodes[i]][nodes[j]]

        # Prim's algorithm
        visited = [False]*n_nodes
        min_edge = [math.inf]*n_nodes
        min_edge[0] = 0
        total_cost = 0

        for _ in range(n_nodes):
            u = -1
            for i in range(n_nodes):
                if not visited[i] and (u == -1 or min_edge[i] < min_edge[u]):
                    u = i

            visited[u] = True
            total_cost += min_edge[u]

            for v in range(n_nodes):
                if not visited[v] and dist_sub[u][v] < min_edge[v]:
                    min_edge[v] = dist_sub[u][v]

        return total_cost

    def solve(self):
        """Giải TSP bằng A*"""

        start_state = (0, 0, (self.start,), self.start)
        pq = []
        heapq.heappush(pq, start_state)

        best_cost = math.inf
        best_path = None
        visited_states = {}

        while pq:
            f, g, visited, current = heapq.heappop(pq)


            state_key = (visited, current)
            if state_key in visited_states and visited_states[state_key] <= g:
                continue
            visited_states[state_key] = g


            if len(visited) == self.n:
total = g + self.dist[current][self.start]
                if total < best_cost:
                    best_cost = total
                    best_path = list(visited) + [self.start]
                continue


            for nxt in range(self.n):
                if nxt in visited:
                    continue

                new_g = g + self.dist[current][nxt]
                if new_g >= best_cost:  # Cắt tỉa
                    continue

                new_visited = visited + (nxt,)
                h = self.heuristic(new_visited, nxt)
                new_f = new_g + h

                heapq.heappush(pq, (new_f, new_g, new_visited, nxt))

        return best_path, best_cost
if __name__ == "__main__":

    dist_matrix = [
        [0, 10, 15, 20],
        [10, 0, 35, 25],
        [15, 35, 0, 30],
        [20, 25, 30, 0]
    ]

    tsp_solver = TSP_AStar(dist_matrix, start=0)
    path, cost = tsp_solver.solve()

    print("Đường đi tối ưu tìm được:", path)
    print("Tổng chi phí:", cost)


Đường đi ngắn nhất từ A đến Goal: ['A', 'C', 'F', 'I', 'L', 'Goal']
Tổng chi phí của đường đi: 11
